# 🔬 RedPitayaSTCL: Setup & Compatibility Check

<p style="font-size:1.05em; color:#444;">
This notebook verifies that your <strong>PC environment</strong> and each
<strong>RedPitaya board</strong> meet all requirements before running the
Scanning Transfer Cavity Lock (STCL) system.
</p>

<blockquote style="border-left:4px solid #e67e22; padding:6px 12px;
  background:#fdf6ec; color:#7f4f00; border-radius:4px;">
  <strong>→ Run all cells top-to-bottom on a fresh kernel before any locking workflow. Remember to update ip-address in the code cell below.</strong>
</blockquote>

<br>

| Step | What it does |
|------|-------------|
| **①** | Configuration — set board IPs here |
| **②** | Import the checker module |
| **③** | PC environment checks |
| **④** | Board information (SSH) |
| **⑤** | Board compatibility checks |
| **⑥** | Summary report |


---
## 1) Configuration

Edit the cell below **before running anything else**. Add one entry per physical RedPitaya board.

| Key | Example | Description |
|-----|---------|-------------|
| `ip` | `"192.168.0.101"` | Board IP address on your network |
| `mode` | `"scan"` | One of: `scan` · `lock` · `monitor` |

<br>

| Mode | Role | Outputs used |
|------|------|-------------|
| `scan` | Generates cavity scan ramp + trigger square wave | OUT2 → ramp, OUT1 → trigger |
| `lock` | Applies PID feedback to laser current mod inputs | OUT1 → Slave1, OUT2 → Slave2 |
| `monitor` | Passive cavity signal monitor (no outputs) | — |


In [7]:
# ── Edit here ────────────────────────────────────────────────────────────────
BOARDS = {
    "Cav"  : {"ip": "192.168.0.200", "mode": "scan"},
    # "Lock1": {"ip": "192.168.0.102", "mode": "lock"},
    # "Mon"  : {"ip": "192.168.0.100", "mode": "monitor"},
}

SSH_USER       = "root"
SSH_PASS       = "root"
SSH_PORT       = 22

STCL_CMD_PORT  = 5000   # RP_Server command listener
STCL_LOOP_PORT = 5065   # reaction_loop port (opened during lock / scan)

---
## 2) Import the Checker Module

All check logic lives in `setup.py` (same directory as this notebook).
The cell below locates the repo root, adds it to `sys.path`, and imports the module.

> **If you see `ModuleNotFoundError: No module named 'setup'`** — make sure
> `setup.py` is in the same folder as this notebook, or add the path manually:
> `sys.path.insert(0, '/path/to/RedPitayaSTCL')`


In [8]:
import sys, pathlib

# Automatically find the repo root (folder containing setup.py)
_nb_dir = pathlib.Path().resolve()
for _candidate in [_nb_dir] + list(_nb_dir.parents)[:3]:
    if (_candidate / "setup.py").exists() and (_candidate / "lockclient.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

import setup                    # RedPitayaSTCL setup.py
import importlib; importlib.reload(setup)   # pick up edits without restarting kernel
print("setup.py loaded from:", pathlib.Path(setup.__file__).resolve())

setup.py loaded from: /Users/devesh/Desktop/Projects/RP-STCL/setup.py


---
## 3) PC Environment Checks

Verifies the Python version, all required packages, the `Qt5Agg` matplotlib
backend, and that the repository is correctly importable. All checks must pass
before connecting to any board.

| Package | Used for |
|---------|----------|
| `paramiko` | SSH/SFTP — upload scripts and start server on each RP |
| `numpy` | Signal processing and array operations |
| `scipy` | Golden-ratio figure sizing; Savitzky-Golay filter math |
| `matplotlib` | Qt5Agg backend for live cavity and error monitor windows |


<br>

> **If Qt5Agg is missing:** `pip install PyQt5`


In [9]:
pc_ok = setup.check_pc()

────────────────────────────────────────────────────────────────────
  PC ENVIRONMENT CHECKS
────────────────────────────────────────────────────────────────────
  ✓ Python version  →  3.12.12 (>= 3.7 required for f-strings and other PC-side syntax)
  ✓ Package: paramiko      →  v4.0.0  —  SSH/SFTP: uploads scripts and starts server on each RP
  ✓ Package: numpy         →  v1.26.4  —  Signal processing and array operations
  ✓ Package: scipy         →  v1.17.1  —  Golden-ratio figure sizing; Savitzky-Golay filter math
  ✓ Package: matplotlib    →  v3.10.8  —  Qt5Agg backend for live cavity and error monitor windows
  ✓ matplotlib backend  →  Qt5Agg available — live monitor windows will work
  ✓ Repo on sys.path  →  /Users/devesh/Desktop/Projects/RP-STCL
  ✓ RP_side importable  →  peak_finders module imported successfully
  ✓ RP_side/RP_Lock.py            →  Main locking loop (uploaded to board)
  ✓ RP_side/RunLock.py            →  Entry point executed on board via SSH
  ✓ RP_side/libse

---
## 4) Board Information

<p>Connects to each board via SSH and collects full hardware and software details
<em>before</em> any pass/fail judgement is applied.
Read the raw output here to understand exactly what is installed on each board.</p>

<blockquote style="border-left:4px solid #2980b9; padding:6px 12px;
  background:#eaf4fb; color:#1a5276; border-radius:4px;">
  <strong>What is collected:</strong>
  OS version · RP ecosystem version · hostname · uptime · CPU · RAM ·
  Python version &amp; sys.path · numpy version · redpitaya pip package ·
  mercury overlay path · STCL files on board · port 5000/5065 status ·
  stale RunLock processes · active system services
</blockquote>

<br>

> **SSH note:** The checker uses `disabled_algorithms` to work around a
> key-exchange incompatibility between newer `paramiko` versions and the
> OpenSSH on Ubuntu 16.04 that ships with RP OS 1.04.


In [10]:
import setup   # ensure latest version

board_infos = {}
for name, cfg in BOARDS.items():
    info = setup.collect_board_info(
        name, cfg["ip"], cfg["mode"],
        ssh_user=SSH_USER, ssh_pass=SSH_PASS, ssh_port=SSH_PORT,
        stcl_cmd_port=STCL_CMD_PORT, stcl_loop_port=STCL_LOOP_PORT,
    )
    board_infos[name] = info

────────────────────────────────────────────────────────────────────
  Board: Cav  (192.168.0.200)  mode=scan
────────────────────────────────────────────────────────────────────
  ✓ Ping 192.168.0.200  — reachable
  ✓ SSH login  root@192.168.0.200
  i OS                : Ubuntu 16.04.7 LTS
  i RP ecosystem      : 1.04
  i RP .version file  : 1.07
  i Hostname          : rp-f0c97c
  i Uptime            : up 1 hour, 10 minutes
  i CPU               : ARMv7 Processor rev 0 (v7l)
  i Memory            : 431 MB total, 36 MB used
  i Python            : Python 3.5.2  (/usr/bin/python3)
  i /home/jupyter/RedPitaya on board sys.path: yes
  i numpy             : 1.11.0
  i redpitaya package : redpitaya (0.98, /home/jupyter/RedPitaya)
  i mercury.py        : /home/jupyter/RedPitaya/redpitaya/overlay/mercury.py
  i mercury importable: yes
  i /opt/redpitaya/bin : present
  i /opt/redpitaya/fpga: present
  ✓ Board STCL file  : /home/jupyter/RedPitaya/RP_Lock.py
  ✓ Board STCL file  : /home/jupyte

---
## 5) Board Compatibility Checks

Evaluates the board information against the known requirements of the STCL
codebase. Each result is marked:

| Symbol | Meaning |
|--------|---------|
| `✓` | **Pass**: requirement met |
| `⚠` | **Warning**: may work but needs attention |
| `✗` | **Fail**: must be resolved before running STCL |

<br>

<details>
<summary><strong>Common failures and fixes (click to expand)</strong></summary>

<br>

| Issue | Fix |
|-------|-----|
| RP OS version > 1.04 | Re-flash SD card with OS 1.04 from `downloads.redpitaya.com` |
| `/home/jupyter/RedPitaya` not on board `sys.path` | Add `export PYTHONPATH=/home/jupyter/RedPitaya:$PYTHONPATH` to `/root/.bashrc` on board |
| numpy ≥ 2.0 on board | Use patched `peak_finders.py` from this repo (`np.mat` → `np.array`) |
| Port 5000 in use | `pkill -f RunLock.py` on the board |
| SCPI service active | `systemctl stop redpitaya_scpi` on the board |
| STCL files not uploaded | Run `LockClient(RPs)` — `upload_current()` transfers them via SFTP |

</details>


In [11]:
board_ok_all = True
for name, info in board_infos.items():
    ok = setup.check_board(info,
                           stcl_cmd_port=STCL_CMD_PORT,
                           stcl_loop_port=STCL_LOOP_PORT)
    board_ok_all = board_ok_all and ok

────────────────────────────────────────────────────────────────────
  Compatibility checks: Cav  (192.168.0.200)
────────────────────────────────────────────────────────────────────
  ✓ OS version  →  Ubuntu 16.04 LTS — matches the tested STCL configuration
  ✓ RP ecosystem version  →  v1.04 — required. STCL README states OS <= 1.04 only.
  ✓ Board Python version  →  Python 3.5.2 — RP-side code requires >= 3.5
  ✓ RedPitaya on board sys.path  →  /home/jupyter/RedPitaya found — mercury overlay will import correctly
  ✓ Board numpy version  →  v1.11.0 — compatible
  ✓ redpitaya pip package  →  redpitaya (0.98, /home/jupyter/RedPitaya)
  ✓ mercury overlay importable  →  /home/jupyter/RedPitaya/redpitaya/overlay/mercury.py
  ✓ /opt/redpitaya/bin  →  present
  ✓ /opt/redpitaya/fpga  →  present
  ✓ Board STCL file: RP_Lock.py
  ✓ Board STCL file: RunLock.py
  ✓ Board STCL file: libserver.py
  ✓ Board STCL file: peak_finders.py
  ✓ Port 5000 free (STCL command)
  ✓ Port 5065 free (STCL loop)

---
## 6) Summary Report

Aggregates all results across PC and boards into a single final verdict.

- All `✗ FAIL` items must be resolved before attempting any locking workflow.
- `⚠ WARNING` items should be reviewed — they may cause subtle issues at runtime.
- A green `✓` at the bottom means the system is ready.

> After resolving any issues, **restart the kernel and re-run all cells**
> to confirm a clean state.


In [12]:
system_ready = setup.print_summary()



════════════════════════════════════════════════════════════════════
  SUMMARY
════════════════════════════════════════════════════════════════════

  ✓ PC environment
    13 passed  |  0 warnings  |  0 failed

  ⚠ Board: Cav  (192.168.0.200)
    16 passed  |  1 warnings  |  0 failed
    Warnings:
        ⚠ Stale RunLock.py process  →  Running PID(s): 2099 bash -c pgrep -a -f RunLock.py 2>/dev/null  — kill with: pkill -f RunLock.py

════════════════════════════════════════════════════════════════════
  ✓  All checks passed — system ready for STCL operation.
════════════════════════════════════════════════════════════════════

